# Week 3, day 4 (morning) — Worksheet 03 SOLUTIONS: facts, dimensions and measures

Executed in the lab image. Every quoted number is what it actually printed.

Question 6 is the one to take to work with you. It is slide 25's discount-rate
question, computed two defensible-looking ways, and they disagree.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 03 — Facts, dimensions and measures. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr = load("enrollment")
tx = load("transaction")
dtype = load("discount_type")

# A first pass at enrollment-grain money, good enough for this sheet.
# Worksheet 07 builds the real one, with the missing-value rules applied.
per_enr = (tx.groupby("enrl_id")
             .agg(tuition_amount=("full_price", "max"),
                  amount_paid=("payment_amount", "sum"),
                  discount_type_id=("discount_type_id", "max"))
             .reset_index())
per_enr = per_enr.merge(
    dtype[["discount_type_id", "discount_amount"]],
    on="discount_type_id", how="left")
per_enr["discount_amount"] = per_enr["discount_amount"].fillna(0.0)
per_enr = per_enr.merge(enr[["enrl_id", "course_id", "cohort_id", "stu_id"]],
                        on="enrl_id", how="left")
per_enr["enrollment_count"] = 1

# Slide 13's sample daily product-store dataset, reproduced exactly.
inventory = pd.DataFrame({
    "date":    ["May 1, 2024"] * 4 + ["May 2, 2024"] * 4,
    "store":   ["Store A", "Store A", "Store B", "Store B"] * 2,
    "product": ["Product X", "Product Y"] * 4,
    "revenue": [10000, 5000, 20000, 8000, 12000, 6000, 18000, 7000],
    "profit":  [2000, 1000, 6000, 1600, 3000, 1200, 5400, 1400],
    "eod_inventory_units": [1200, 600, 800, 500, 1150, 550, 850, 450],
    "profit_margin_pct": [20.0, 20.0, 30.0, 20.0, 25.0, 20.0, 30.0, 20.0],
})

print("per_enr:  ", per_enr.shape)
print("inventory:", inventory.shape, "(slide 13's sample dataset)")

PART A — sorting the columns

### Question 1

Apply slide 16's test to the `enrollment` table. For each column print the name, its dtype, its distinct count, and your classification: `key`, `measure`, `attribute` or `degenerate`.
> **NOTE:** `enrl_id` identifies the fact row itself. It is not a foreign key to anything and it is not a measure — it is a *degenerate dimension*, and it stays in the fact table.

In [ ]:
CLASS = {
    "enrl_id":   ("degenerate", "identifies the fact row; no dimension table"),
    "enrl_date": ("key", "-> dim_date"),
    "stu_id":    ("key", "-> dim_student"),
    "course_id": ("key", "-> dim_course"),
    "cohort_id": ("key", "-> dim_cohort"),
    "status":    ("attribute", "a reporting rule, not a measure -- see ws06"),
}
for col in enr.columns:
    kind, note = CLASS[col]
    print("  %-11s %-6s %5d distinct   %-11s %s"
          % (col, enr[col].dtype, enr[col].nunique(), kind, note))
print()
print("measures in the source enrollment table:", 0)

```
  enrl_id     int64   2400 distinct   degenerate  identifies the fact row; no dimension table
  enrl_date   str      650 distinct   key         -> dim_date
  stu_id      int64    602 distinct   key         -> dim_student
  course_id   int64     24 distinct   key         -> dim_course
  cohort_id   int64     16 distinct   key         -> dim_cohort
  status      str        2 distinct   attribute   a reporting rule, not a measure -- see ws06

measures in the source enrollment table: 0
```

**Not one measure.** Six columns: one row identifier, four keys, one status flag.

That is worth sitting with, because `fact_enrollment` on slide 29 has five
measures and a flag — `enrollment_count`, `tuition_amount`, `discount_amount`,
`net_tuition_amount`, `amount_paid_to_date`, `is_paid_in_full` — and **none of
them exist in the source table the fact table is built from.** Every one is
either derived (`enrollment_count` is literally the constant 1) or summarised up
from `transaction`. Worksheet 07 creates all six.

So a fact table is not a copy of an event table with the names changed. It is
built.

Two of the classifications are worth defending.

**`enrl_id` is a degenerate dimension.** It identifies the fact row, it is not a
number you would ever sum, and it has no dimension table because there are no
attributes to put in one — everything about the enrollment is already a key or a
measure. Kimball's term for this is a *degenerate dimension*, and the right
handling is to keep it in the fact table as a plain column. It is what makes a
fact row traceable to the source record it came from, which is exactly what you
need when a number looks wrong at 4pm.

**`status` is not a measure**, even though `COUNT(*) WHERE status='cancelled'`
gives a number. The column holds a business state with two values; whether it
becomes a filter, a flag on the fact, or an attribute on a dimension is the
decision worksheet 06 makes.

The distinct counts also preview the dimensions: 650 dates, 602 students, 24
courses, 16 cohorts. Those are the row counts of four of the six dimension tables
worksheet 08 builds.

### Question 2

Fact tables grow; dimension tables mostly do not. For each source table print its row count and the number of rows added in the last 90 days of `enrl_date`, using each table's own date column where it has one.
> **NOTE:** `growth over time` is the practical test for "is this a fact or a dimension". Tables with no date column at all are almost always dimensions.

In [ ]:
cut = pd.to_datetime(enr.enrl_date).max() - pd.Timedelta(days=90)
print("cutoff:", cut.date(), "\n")
DATECOL = {"enrollment": "enrl_date", "transaction": "trans_dt",
           "cohort": "start_dt", "program": "start_date",
           "category": "start_date"}
for name in ["category", "city", "cohort", "course", "discount_type",
             "employee", "employee_type", "enrollment", "payment_type",
             "program", "students", "transaction"]:
    df = load(name)
    col = DATECOL.get(name)
    if col:
        recent = int((pd.to_datetime(df[col]) > cut).sum())
        print("  %-14s %5d rows   %5d in last 90 days" % (name, len(df), recent))
    else:
        print("  %-14s %5d rows   no date column" % (name, len(df)))

```
  category           5 rows       0 in last 90 days
  city              30 rows   no date column
  cohort            16 rows       3 in last 90 days
  course            24 rows   no date column
  discount_type      6 rows   no date column
  employee          14 rows   no date column
  employee_type      3 rows   no date column
  enrollment      2400 rows     344 in last 90 days
  payment_type       5 rows   no date column
  program            8 rows       0 in last 90 days
  students         606 rows   no date column
  transaction     4856 rows     736 in last 90 days
```

Two tables grew in the last 90 days in any meaningful way — **`enrollment` (344
rows) and `transaction` (736)**. `cohort` added 3, which is a cohort being
scheduled rather than an event occurring. The other nine added nothing, and seven
of them have no date column at all.

That is the practical fact/dimension test, and it beats the definitional one
because you can run it on a schema you have never seen. **A table that records
when its rows happened, and keeps getting more of them, is a fact table.
A table that describes things, and changes only when the business changes, is a
dimension.**

`enrollment` and `transaction` are the two candidate fact tables. Slide 29 builds
`fact_enrollment` from the first and summarises the second into it; slide 29's own
note says a separate `fact_payment` from the second would turn the model into a
galaxy schema. Worksheet 04 builds that.

Two edges worth noticing.

**`cohort` is ambiguous** — it has dates and it grows, just very slowly, and the
dates describe a *scheduled period* rather than a recorded event. It is a
dimension, and it is the kind of table that generates an argument. The tiebreaker
is the grain: `fact_enrollment` has one row per enrollment, and a cohort is
something an enrollment *has*, not something that happens.

**`students` has no date column**, and it is a dimension with 606 rows. But
students *do* accumulate — a school with 600 students today has 6,000 in ten
years. Dimensions that grow are perfectly normal; the distinction is not size or
growth but whether a row records an **event** or describes a **thing**.

PART B — additive

### Question 3

Test `enrollment_count` and `amount_paid` for additivity: total them over the whole table, then re-total them after grouping by `course_id`, by `cohort_id`, and by both. Print all the totals.

In [ ]:
for measure in ("enrollment_count", "amount_paid"):
    whole = per_enr[measure].sum()
    by_course = per_enr.groupby("course_id")[measure].sum().sum()
    by_cohort = per_enr.groupby("cohort_id")[measure].sum().sum()
    by_both = per_enr.groupby(["course_id", "cohort_id"])[measure].sum().sum()
    print("%s:" % measure)
    print("  ungrouped          %15.2f" % whole)
    print("  by course          %15.2f" % by_course)
    print("  by cohort          %15.2f" % by_cohort)
    print("  by course + cohort %15.2f" % by_both)
    print("  all equal:", len({round(whole, 2), round(by_course, 2),
                               round(by_cohort, 2), round(by_both, 2)}) == 1)
    print()

```
enrollment_count:
  ungrouped                  2243.00
  by course                  2243.00
  by cohort                  2243.00
  by course + cohort         2243.00
  all equal: True

amount_paid:
  ungrouped               8953821.53
  by course               8953821.53
  by cohort               8953821.53
  by course + cohort      8953821.53
  all equal: True
```

Both survive every regrouping unchanged, to the cent. That is what **additive**
means, and this is the test — group the measure any way you like, sum the groups,
and the total must not move.

Slide 11 defines additive as *"can be summed across all relevant dimensions at
the defined grain"*, and this output is that definition made executable. It is
worth running on any measure you are about to expose to a BI tool, because a BI
tool will let a user drag that column onto any axis and will happily sum it
there.

Two notes on the numbers.

**2,243, not 2,400.** `per_enr` is built from `transaction`, so the 157
enrollments with no payment are missing — the inner-join filter from worksheet 01
question 7, still in effect. Worksheet 07 fixes it with a left join. Nothing
about additivity is affected, but it is a good reminder that a measure can be
perfectly additive and still be computed over the wrong set of rows.

**`enrollment_count` is a constant 1.** It looks pointless and it is not. Slide 28
lists it as a measure and slide 37 says *"assign 1 per enrollment record"*, for
one reason: it makes counting an addition. `SUM(enrollment_count)` works
identically whether the fact table is queried raw, pre-aggregated into a summary,
or rolled up by a BI tool — whereas `COUNT(*)` breaks the moment anything
upstream aggregates, because it starts counting summary rows. Worksheet 02
question 9 is what that failure looks like.

PART C — semi-additive

### Question 4

Use slide 13's sample dataset. Print it, then total `revenue` and `eod_inventory_units` per store per date, and then across both dates. Compare what each total means.
> **NOTE:** slide 13's own conclusion — 1,800 + 1,300 = 3,100 units is valid; 1,800 + 1,700 = 3,500 is not. Reproduce both.

In [ ]:
print(inventory.to_string(index=False))
print()
per = inventory.groupby(["date", "store"])[["revenue", "eod_inventory_units"]].sum()
print(per.to_string())
print()
print("across STORES, on one date:")
for d in inventory.date.unique():
    sub = inventory[inventory.date == d]
    print("  %-12s revenue %6d   inventory %5d units"
          % (d, sub.revenue.sum(), sub.eod_inventory_units.sum()))
print()
print("across DATES, for Store A:")
a = inventory[inventory.store == "Store A"]
print("  revenue   %6d  <- meaningful: two days of sales" % a.revenue.sum())
print("  inventory %6d  <- NOT meaningful: a stock level counted twice"
      % a.eod_inventory_units.sum())

Slide 13's dataset, exactly as printed on the slide:

```
       date   store   product  revenue  profit  eod_inventory_units  profit_margin_pct
May 1, 2024 Store A Product X    10000    2000                 1200               20.0
May 1, 2024 Store A Product Y     5000    1000                  600               20.0
May 1, 2024 Store B Product X    20000    6000                  800               30.0
May 1, 2024 Store B Product Y     8000    1600                  500               20.0
May 2, 2024 Store A Product X    12000    3000                 1150               25.0
...
```

Rolled up per store per date:

```
                     revenue  eod_inventory_units
date        store
May 1, 2024 Store A    15000                 1800
            Store B    28000                 1300
May 2, 2024 Store A    18000                 1700
            Store B    25000                 1300
```

Now the two directions.

**Across stores, on one date** — this is fine:

```
  May 1, 2024  revenue  43000   inventory  3100 units
  May 2, 2024  revenue  43000   inventory  3000 units
```

3,100 units is a real quantity: the stock standing in both stores at the end of
1 May. Slide 13 says the same — *1,800 + 1,300 = 3,100 units, valid on May 1*.

**Across dates, for one store** — this is where they part company:

```
  revenue    33000  <- meaningful: two days of sales
  inventory   3500  <- NOT meaningful: a stock level counted twice
```

Revenue over two days is two days of sales, and 33,000 is a fact about the week.
Inventory over two days is **not** two days of stock. Store A did not have 3,500
units; it had 1,800 at the end of Monday and 1,700 at the end of Tuesday, and
those are largely *the same physical units counted twice*.

That is **semi-additive**: additive across store and product, not additive across
time. And the tell is what the measure *is* — a **snapshot** of a level at a
point in time, rather than a **flow** accumulated over a period. Account balance,
headcount, stock on hand, open tickets: all snapshots, all semi-additive.

The dangerous part is that nothing in the data marks the difference. `revenue`
and `eod_inventory_units` are both integers in adjacent columns, and `SUM` works
identically on both. 3,500 does not throw an error, look odd, or fail a
reconciliation. It is simply not a thing.

Question 5 is what to do instead.

### Question 5

Show the right way to roll a semi-additive measure over time. For `eod_inventory_units` per store, print the sum, the average across dates, and the value on the latest date. Say which one you would put in a report.

In [ ]:
g = inventory.groupby(["store", "date"])["eod_inventory_units"].sum()
print(g.to_string())
print()
latest = inventory.date.max()
for store in sorted(inventory.store.unique()):
    s = g.loc[store]
    print("  %-8s  SUM %5d (wrong)   AVG %6.1f   LATEST(%s) %5d"
          % (store, s.sum(), s.mean(), latest,
             inventory[(inventory.store == store)
                       & (inventory.date == latest)].eod_inventory_units.sum()))

```
store    date
Store A  May 1, 2024    1800
         May 2, 2024    1700
Store B  May 1, 2024    1300
         May 2, 2024    1300

  Store A   SUM  3500 (wrong)   AVG 1750.0   LATEST(May 2, 2024)  1700
  Store B   SUM  2600 (wrong)   AVG 1300.0   LATEST(May 2, 2024)  1300
```

Three answers, and slide 13's own guidance — *"use ending inventory or average
inventory over time instead"* — endorses the second and third.

**`SUM` is never right** over time for a snapshot measure. 3,500 and 2,600 are
not quantities of anything.

**`AVG` (1,750 and 1,300)** answers *"how much stock did we typically hold?"* —
the right measure for questions about carrying cost, warehouse capacity, or
working capital tied up.

**`LATEST` (1,700 and 1,300)** answers *"how much stock do we have?"* — the right
measure for a balance-sheet position or a reorder decision.

Note that for Store B all three "time" answers agree at 1,300 because its level
never moved. That is a useful warning: a stable series makes wrong aggregation
look right, and the method only reveals itself when the numbers change.

Two practical consequences for the model.

**Name the column so the aggregation is obvious.** `eod_inventory_units` is a
good name — "eod" says snapshot. `inventory` alone invites a `SUM`.

**Say it where the tool will see it.** Most BI tools let you set a default
aggregation per measure and a distinct one for the time dimension — Analysis
Services calls it `LastNonEmpty`. If your platform cannot express that, the
honest fallback is to not expose the raw column at all, and publish
`avg_inventory_units` and `closing_inventory_units` as two separate, correctly
aggregatable measures.

Our enrollment model has no semi-additive measure, which is why this question
borrows slide 13's data. But `amount_paid_to_date` is one step away from being
one: as a running total captured daily it would be a snapshot, and summing it
across dates would double-count every payment. It survives as additive here only
because the fact table holds a single row per enrollment rather than a daily
history.

PART D — non-additive

### Question 6

Slide 25 asks *"which course-cohort groups have the highest discount rates?"*. Compute a discount rate per enrollment (`discount_amount / tuition_amount`), then produce a rate per **course-cohort group** two ways: the mean of the per-enrollment rates, and `SUM(discount_amount) / SUM(tuition_amount)`. Print both for the top five groups, and the two overall figures.
> **NOTE:** these are not two roundings of one number. They answer different questions.

In [ ]:
d = per_enr.copy()
d["rate"] = d["discount_amount"] / d["tuition_amount"]
grp = d.groupby(["course_id", "cohort_id"]).agg(
    n=("enrl_id", "size"),
    mean_of_rates=("rate", "mean"),
    discount=("discount_amount", "sum"),
    tuition=("tuition_amount", "sum"))
grp["ratio_of_sums"] = grp.discount / grp.tuition

print("course-cohort groups:", len(grp))
print()
print("the five with the highest discount rate:")
print(grp.sort_values("mean_of_rates", ascending=False)
      [["n", "mean_of_rates", "ratio_of_sums"]].head(5).round(4).to_string())
print()
print("overall, mean of per-enrollment rates: %.4f" % d["rate"].mean())
print("overall, SUM(discount)/SUM(tuition):   %.4f"
      % (d.discount_amount.sum() / d.tuition_amount.sum()))

```
course-cohort groups: 383

the five with the highest discount rate:
                     n  mean_of_rates  ratio_of_sums
course_id cohort_id
223       308        1         0.5714         0.5714
213       311        4         0.3321         0.3103
207       302        3         0.2730         0.2560
219       316        2         0.2381         0.2222
207       314        4         0.2345         0.2365

overall, mean of per-enrollment rates: 0.0875
overall, SUM(discount)/SUM(tuition):   0.0835
```

Two methods, two answers, everywhere. Overall: **0.0875** against **0.0835**.

Neither is a rounding of the other, and neither is wrong. They answer different
questions:

- **mean of rates** — "what discount does a typical enrollment get?" Every
  enrollment counts once, whatever its tuition.
- **ratio of sums** — "what fraction of our tuition did we give away?" Every
  dollar counts once, so expensive enrollments weigh more.

For slide 25's question — *"which course-cohort groups have the highest discount
rates?"* — the second is almost certainly what is meant, because the concern
behind it is money. But the deck does not say, and that is the point: **the
formula is a business decision, not a technical one**, and it has to be written
down where the next person will find it.

This is slide 11's **non-additive** row: *"should not be summed directly —
ratio, percentage, average discount rate"*. Note the third example is precisely
this column. A rate cannot be summed, and it also cannot be safely averaged,
because averaging is summing with a division on the end.

**The rule that follows is the important part of this worksheet.** Do not store
`discount_rate` in the fact table. Store `discount_amount` and `tuition_amount`
— both additive — and compute the rate in the query, at whatever grain the
question needs:

```sql
SELECT course_id, cohort_id,
       SUM(discount_amount) / SUM(tuition_amount) AS discount_rate
FROM fact_enrollment
GROUP BY course_id, cohort_id;
```

Store the numerator and the denominator; derive the ratio. A rate stored in a
fact table is correct at exactly one grain and silently wrong at every other, and
nothing about the column says which grain that was.

Now look at the top row of that ranking: **n = 1**. Question 7.

### Question 7

Look hard at the `n` column in question 6's ranking. Print the distribution of group sizes, how many groups have three enrollments or fewer, and the average of the 383 group rates against the true overall rate.
> **NOTE:** a rate is a ratio, and a ratio over a small denominator is mostly noise. Check what is behind the top of the ranking before reporting it.

In [ ]:
d = per_enr.copy()
d["rate"] = d["discount_amount"] / d["tuition_amount"]
grp = d.groupby(["course_id", "cohort_id"]).agg(n=("enrl_id", "size"),
                                                mean_rate=("rate", "mean"))
print("enrollments per course-cohort group:")
print(grp.n.describe()[["min", "25%", "50%", "75%", "max"]].to_string())
print()
print("groups with 3 enrollments or fewer:", int((grp.n <= 3).sum()),
      "of", len(grp))
print("enrollments behind the top-ranked group:", int(
    grp.sort_values("mean_rate", ascending=False).n.iloc[0]))
print()
true_rate = d.discount_amount.sum() / d.tuition_amount.sum()
print("average of the %d group rates: %.4f" % (len(grp), grp.mean_rate.mean()))
print("true overall discount rate:    %.4f" % true_rate)
print()
big = grp[grp.n >= 8]
print("the five highest rates among groups with 8+ enrollments:")
print(big.sort_values("mean_rate", ascending=False).head(5).round(4).to_string())

```
enrollments per course-cohort group:
min     1.0
25%     4.0
50%     6.0
75%     7.0
max    15.0

groups with 3 enrollments or fewer: 62 of 383
enrollments behind the top-ranked group: 1

average of the 383 group rates: 0.0867
true overall discount rate:     0.0835
```

**The group with the highest discount rate in the business has one enrollment in
it.**

Its rate of 0.5714 is not a discounting pattern. It is one student who took one
scholarship, reported as a course-cohort with a rate more than six times the company
average of 0.0835. And it is not an isolated case: **62 of the 383 groups — 16% — have
three enrollments or fewer**, and every one of them can reach the top of this
ranking on a single unusual row.

Filtering to groups with a real denominator gives a completely different, and
actually useful, answer:

```
the five highest rates among groups with 8+ enrollments:
                      n  mean_rate
course_id cohort_id
218       302         8     0.2221
          314        11     0.2149
201       315         9     0.2049
213       304         8     0.1701
203       303         8     0.1673
```

Course 218 now appears **twice**, in two different cohorts, at 22.2% and 21.5%.
That is a pattern — something about how course 218 is sold — and it is a lead
worth following. The n=1 group at 0.5714 was noise wearing a percentage sign.

Slide 25's last question is *"which course-cohort groups may be over-discounted
or underperforming?"*, and slide 30 answers it with *"compare... against
course-level benchmarks"*. This is why a benchmark is needed: a raw ranking of
383 ratios is a ranking of small denominators.

Two things to take away.

**Never rank by a ratio without showing its denominator.** Put `n` in the output,
next to the rate, always. It costs one column and it is the difference between a
finding and an artefact.

**Set a minimum group size, and say what it is.** Eight is arbitrary — the honest
version is to agree a threshold with the business and record it in the metric
definition. What is not acceptable is publishing the unfiltered ranking, because
the top of it is guaranteed to be the smallest groups.

Note also the average of the 383 group rates: **0.0867 against a true 0.0835**.
Averaging group averages weights a group of 1 the same as a group of 15. It is a
third answer, and it is the one nobody intends.

PART E — why attributes stay out of the fact table

### Question 8

Suppose `course_name` went into the fact table instead of `dim_course`. Build that column onto `per_enr` and print: the number of rows storing each name, the total characters stored, and the same figure for a dimension table holding each name once.

In [ ]:
crs = load("course")[["course_id", "course_name"]]
wide = per_enr.merge(crs, on="course_id", how="left")

in_fact = wide["course_name"].str.len().sum()
in_dim = crs["course_name"].str.len().sum()
print("fact rows:                 ", len(wide))
print("distinct course names:     ", crs.course_name.nunique())
print()
print("characters if stored in the fact table: %8d" % in_fact)
print("characters if stored in dim_course:     %8d" % in_dim)
print("ratio: %.1fx" % (in_fact / in_dim))
print()
print("rows to UPDATE if one course is renamed:")
top = wide.course_name.value_counts().head(3)
print(top.rename("fact rows").to_string())
print("  ...in dim_course: 1")

```
fact rows:                  2243
distinct course names:      24

characters if stored in the fact table:    50999
characters if stored in dim_course:          547
ratio: 93.2x
```

**93 times the storage** for the same 24 pieces of information — and at 2,243
rows this is a toy. A real fact table has hundreds of millions of rows, several
text attributes, and the ratio scales with it.

But storage is the least of it. The other consequence:

```
rows to UPDATE if one course is renamed:
course_name
Distributed Systems Primer    114
Streaming Fundamentals        114
Feature Engineering           113
  ...in dim_course: 1
```

Rename one course and you rewrite **114 fact rows** instead of one dimension row.
That is an *update anomaly*, and its real cost is not the write — it is that the
update can partially fail, or be run twice, or miss rows added since. Then 100
rows say the new name and 14 say the old one, both spellings appear in every
report grouped by course, and the totals split.

A dimension table makes that structurally impossible. There is one row holding
the name, so there is nothing for it to disagree with.

Slide 16's rule — fact tables hold *"measures and foreign keys"*, dimensions hold
*"attributes and hierarchies"* — is usually taught as a query-simplicity idea.
These two numbers are the engineering reason underneath it.

Three points of nuance, because the rule is not absolute.

**Degenerate dimensions stay.** `enrl_id` is text-like and belongs in the fact
table, because there are no attributes to factor out (question 1).

**Columnar storage changes the storage arithmetic, not the anomaly.** Snowflake,
BigQuery and Parquet all dictionary-encode a low-cardinality string column, so
24 distinct names across 2,243 rows compress to nearly nothing. The 93x is close
to a worst case. The update anomaly survives compression entirely.

**Some teams denormalise deliberately anyway** — a "one big table" model, where
`course_name` sits on the fact row to avoid a join. That is a real and sometimes
correct choice on modern warehouses. What makes it defensible is doing it on
purpose, with a rebuild-from-dimensions process, rather than because an attribute
drifted into the fact table and nobody moved it out.

### Question 9

Dimensions carry hierarchies. Build the `category -> program -> course` chain and print, for each level, the number of distinct values, then show the full path for three courses.
> **NOTE:** slide 15 calls these out — `region -> store`, `category -> product`. They are why a dimension can be drilled into.

In [ ]:
crs, prg, cat = load("course"), load("program"), load("category")
chain = (crs[["course_id", "course_name", "program_id"]]
         .merge(prg[["program_id", "program_name", "category_id"]], on="program_id")
         .merge(cat[["category_id", "category_name"]], on="category_id", how="left"))
chain["category_name"] = chain["category_name"].fillna("Unknown")
print("levels in the hierarchy:")
for col in ("category_name", "program_name", "course_name"):
    print("  %-14s %3d distinct" % (col, chain[col].nunique()))
print()
print(chain[["category_name", "program_name", "course_name"]]
      .sort_values("course_name").head(3).to_string(index=False))

```
levels in the hierarchy:
  category_name    5 distinct
  program_name     8 distinct
  course_name     24 distinct

  category_name               program_name                    course_name
Cloud Computing Cloud Platform Engineering Analytics Engineering with dbt
        Unknown       Foundations Bootcamp               Capstone Project
  Cybersecurity        Security Operations             Career Foundations
```

5 categories, 8 programs, 24 courses — a widening hierarchy, which is what slide
15 means by *"hierarchies, such as region → store or category → product"*.

The reason it matters is drill-down. A report showing enrollments by category has
5 rows; clicking one shows its programs; clicking one of those shows its courses.
That whole interaction is possible only because the three levels sit in **one
dimension table**, so the BI tool can move between them without a join.

Which is the decision worksheet 04 examines. This hierarchy could be stored two
ways:

- **flattened into `dim_course`** — one table with `course_name`, `program_name`,
  `category_name` side by side, `category_name` repeated across all 24 rows.
  That is a **star**.
- **split into `dim_course` → `dim_program` → `dim_category`** — three tables,
  each value stored once, two extra joins to get a category. That is a
  **snowflake**.

Slide 29 chooses neither exactly: it puts `program_category` on `dim_program`,
and gives `dim_program` and `dim_course` each their own key on the fact table.
Worksheet 08 builds that and worksheet 04 measures what the alternatives cost.

Notice `Unknown` in the second row. That is the null `category_name` from
worksheet 01 question 4, handled with `fillna("Unknown")` — the first appearance
of slide 36's *"missing category → Unknown"* rule. Worksheet 06 applies it
properly. Without it, the `Foundations Bootcamp` program and its 316 enrollments
vanish from every grouped report, silently.

Then the third row: `Career Foundations` sits under `Security Operations`, which
sits under `Cybersecurity`. A career-skills course inside a security program is
the sort of thing a hierarchy makes visible and a flat list does not — a question
for the business, not a defect.

### Question 10

Finally, treat a dimension attribute as a measure: run `enr_named["course_name"].mean()` after joining the names on. **This is supposed to fail.** Read the error and say what it is really objecting to.

In [ ]:
crs = load("course")[["course_id", "course_name"]]
enr_named = per_enr.merge(crs, on="course_id", how="left")
print("course_name dtype:", enr_named["course_name"].dtype)
print("a sample value:   ", repr(enr_named["course_name"].iloc[0]))
print()
print(enr_named["course_name"].mean())

```
course_name dtype: str
a sample value:    'Python for Data Engineering'

TypeError: Cannot perform reduction 'mean' with string dtype
```

pandas refuses, and it is refusing on the grounds of the **dtype** — not on the
grounds that the question is meaningless. That distinction is the last thing this
worksheet has to say.

"What is the average course name?" is a category error, and it got caught only
because course names happen to be text. Rerun the same mistake on a numeric
attribute and nothing stops you:

- `AVG(course_id)` returns about 212.5. It is a number. It means nothing.
- `SUM(cohort_id)` returns a large integer. Also meaningless.
- `AVG(discount_rate)` returns 0.0875 — meaningless in the specific way question
  6 established, and utterly indistinguishable from a correct answer.

**The type system catches the silly version of this error and none of the
serious ones.** Which is why the fact/dimension split is a modelling discipline
rather than something a database can enforce: by the time a column is a `FLOAT`
in a fact table, nothing downstream knows whether it is additive, semi-additive,
non-additive, or a key that happens to be numeric.

**What this sheet established:**

| | |
|---|---|
| the source `enrollment` table | 4 keys, 1 degenerate key, 1 attribute, **0 measures** — all six fact measures are built |
| additive | `enrollment_count` and `amount_paid` — identical under every regrouping |
| semi-additive | inventory: 1,800 + 1,300 = 3,100 valid across stores; 1,800 + 1,700 = 3,500 meaningless across dates |
| non-additive | discount rate: **0.0875 or 0.0835** depending on method, both defensible |
| ranking by a ratio | the top course-cohort group has **n = 1** and a rate of **0.5714** |
| an attribute in the fact table | **93.2x** the storage, and 114 rows to update instead of 1 |

The practical residue is three rules:

1. **Store the numerator and denominator, derive the ratio.** Never store a rate
   in a fact table.
2. **Show the denominator whenever you show a rate**, and set a minimum group
   size before ranking.
3. **Record each measure's aggregation rule** — additive, semi-additive with its
   time treatment, or non-additive and not to be aggregated — as part of the
   model, because neither the column type nor the column name carries it.

Worksheet 04 turns to the shape of the schema itself: star, snowflake, and
galaxy, with the joins and row counts measured rather than described.